<a href="https://colab.research.google.com/github/ElNJoel/Super-Mario-World-Remastered-3DS/blob/main/Notebooks/finetune/xtts/colab_xtts_finetune_webui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Welcome to the *xtts*-finetune-webui gradio gui!

This webui is a slightly modified copy of the official webui for finetune xtts.

If you are looking for an option for normal XTTS use look here https://github.com/daswer123/xtts-webui

In [1]:
# @title 🛠️ Install requirments
#!DEBIAN_FRONTEND=noninteractive
!sudo apt-get update # && sudo apt-get -y upgrade
!sudo apt-get -y install libegl1
!sudo apt-get -y install libopengl0
!sudo apt-get -y install libxcb-cursor0
!pip install -r https://raw.githubusercontent.com/daswer123/xtts-finetune-webui/main/requirements.txt
!pip install gradio==4.44.1
!pip install fastapi==0.103.1
!pip install pydantic==2.3.0

#More requirements
!nvcc --version #check cuda version if things break
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,899 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [102 kB]
Get:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 https://

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Found existing installation: torch 2.1.2
Uninstalling torch-2.1.2:
  Successfully uninstalled torch-2.1.2
Found existing installation: torchvision 0.16.2
Uninstalling torchvision-0.16.2:
  Successfully uninstalled torchvision-0.16.2
Found existing installation: torchaudio 2.1.2
Uninstalling torchaudio-2.1.2:
  Successfully uninstalled torchaudio-2.1.2
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 GB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 23.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.0 MB/s eta 0:00:00


In [ ]:
# @title 🚀 Run interface
%cd /content/
!git clone https://github.com/DrewThomasson/xtts-finetune-webui.git
%cd /content/xtts-finetune-webui
!python xtts_demo.py --share

/content
Cloning into 'xtts-finetune-webui'...
remote: Enumerating objects: 285, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 285 (delta 113), reused 85 (delta 81), pack-reused 149 (from 3)
Receiving objects: 100% (285/285), 136.43 KiB | 9.09 MiB/s, done.
Resolving deltas: 100% (144/144), done.
/content/xtts-finetune-webui
Running on local URL:  http://127.0.0.1:5003
Running on public URL: https://f1c8f4699a511b372d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
config.json: 2.39kB [00:00, 15.6MB/s]
preprocessor_config.json: 100% 340/340 [00:00<00:00, 3.09MB/s]
tokenizer.json: 0.00B [00:00, ?B/s]
vocabulary.json: 1.07MB [00:00, 43.0MB/s]
tokenizer.json: 2.48MB [00:00, 66.0MB/s]
model.bin: 100% 3.09G/3.09G [00:32<00:00, 96.2MB/s]
Warning, existing language does not match target langua

In [ ]:
import shutil
import requests
import os
from tqdm import tqdm  # Progress bar library

# Define the paths
finetune_dir = '/content/xtts-finetune-webui/finetune_models/ready'  # @param {type:"string"}
dataset_dir = '/content/xtts-finetune-webui/finetune_models/dataset'  # @param {type:"string"}

# Create a temporary directory to collect both folders before zipping
temp_dir = "/content/temp_finetune_dataset"
os.makedirs(temp_dir, exist_ok=True)

# Copy both directories into the temporary directory with a progress bar
def copy_with_progress(src, dst):
    total_files = sum(len(files) for _, _, files in os.walk(src))
    with tqdm(total=total_files, desc=f"Copying {os.path.basename(src)}") as pbar:
        for root, _, files in os.walk(src):
            rel_path = os.path.relpath(root, src)
            target_path = os.path.join(dst, rel_path)
            os.makedirs(target_path, exist_ok=True)
            for file in files:
                shutil.copy(os.path.join(root, file), target_path)
                pbar.update(1)

copy_with_progress(finetune_dir, os.path.join(temp_dir, "ready"))
copy_with_progress(dataset_dir, os.path.join(temp_dir, "dataset"))

# Create a zip file of the combined directories with progress
zip_filename = "finetune_and_dataset.zip"
with tqdm(total=100, desc="Zipping files") as pbar:
    shutil.make_archive("finetune_and_dataset", 'zip', root_dir=temp_dir)
    pbar.update(100)

# Define a function to stream the upload with a progress bar
def upload_with_progress(file_path, url):
    file_size = os.path.getsize(file_path)
    with open(file_path, 'rb') as f, tqdm(
        total=file_size, unit='B', unit_scale=True, desc="Uploading"
    ) as progress:
        response = requests.post(
            url,
            files={"file": (file_path, f)},
            stream=True,
            headers={"Connection": "keep-alive"},
        )
        # Update the progress bar as chunks are sent
        for chunk in response.iter_content(chunk_size=4096):
            if chunk:
                progress.update(len(chunk))
    return response

# Upload the zip file to file.io with a progress bar
response = upload_with_progress(zip_filename, "https://file.io/?expires=1d")

# Parse the response and display the download link
if response.status_code == 200:
    download_link = response.json().get('link', 'Error: No link found.')
    print(f"Your file is ready: {download_link}")
else:
    print(f"Failed to upload: {response.status_code} - {response.text}")
